In [1]:
import os
import sys
import re
# =========================================================
# 1. ÉP VERSION PYSPARK (PHẢI CHẠY ĐẦU TIÊN TRƯỚC KHI IMPORT)
# =========================================================
modules_to_remove = [mod for mod in sys.modules if mod.startswith('pyspark') or mod.startswith('py4j')]
for mod in modules_to_remove: 
    del sys.modules[mod]

sys.path = [p for p in sys.path if "/usr/local/spark" not in p]
if "PYTHONPATH" in os.environ: 
    del os.environ["PYTHONPATH"]
    
conda_site_packages = "/opt/conda/lib/python3.13/site-packages"
if conda_site_packages not in sys.path: 
    sys.path.insert(0, conda_site_packages)
    
os.environ["SPARK_HOME"] = os.path.join(conda_site_packages, "pyspark")
os.environ["PYSPARK_PYTHON"] = "python3"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python3"


# =========================================================
# 2. BÂY GIỜ MỚI IMPORT PYSPARK VÀ KHỞI TẠO SESSION
# =========================================================
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf, concat_ws
from pyspark.sql.types import IntegerType, StringType
from pyspark.ml.feature import StopWordsRemover, CountVectorizer
from pyspark.ml.clustering import LDA
spark = SparkSession.builder \
    .appName("LSTM_word2vec_token_Pipeline") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "2g") \
    .config("spark.driver.maxResultSize", "2g") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "dataNLPmining-lab") \
    .config("spark.hadoop.fs.s3a.secret.key", "dataNLPmining-lab") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.my_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.my_catalog.type", "hadoop") \
    .config("spark.sql.catalog.my_catalog.warehouse", "s3a://raw-financial-data/iceberg_warehouse") \
    .getOrCreate()


# =========================================================
# 3. VÁ LỖI THỜI GIAN HADOOP
# =========================================================
hadoop_conf = spark._jsc.hadoopConfiguration()
iterator = hadoop_conf.iterator()
while iterator.hasNext():
    entry = iterator.next()
    val = str(entry.getValue()).strip().lower()
    match = re.fullmatch(r"(\d+)([smhd])", val)
    if match:
        num, unit = int(match.group(1)), match.group(2)
        ms_val = num * 1000 if unit == 's' else num * 60000 if unit == 'm' else num * 3600000 if unit == 'h' else num * 86400000
        hadoop_conf.set(entry.getKey(), str(ms_val))

print("✅ Khởi tạo Spark và môi trường hoàn tất!")

✅ Khởi tạo Spark và môi trường hoàn tất!


In [2]:
# ==========================================
# 0. CẤU HÌNH THAM SỐ LDA
# ==========================================
NUM_TOPICS = 10        # Số lượng chủ đề bạn muốn phân chia (K)
MAX_ITER = 20          # Số vòng lặp huấn luyện
NUM_KEYWORDS = 5       # Số lượng từ khóa dùng để "tóm tắt" mỗi chủ đề


In [3]:
# Đọc bảng chứa dữ liệu tokenized
df = spark.table("my_catalog.processed_zone_finnhub.news_tokens_sentiment_labeled").dropna(
    subset=["title_tokens", "summary_tokens"]
)

# Để LDA chạy hiệu quả, PySpark cần đầu vào là ArrayType (Mảng các từ)
# Chuyển đổi cột string tokenized thành dạng mảng
from pyspark.sql.functions import split
df = df.withColumn("tokens_array", split(concat_ws(" ", col("title_tokens"), col("summary_tokens")), " "))



In [4]:
# ==========================================
# 2. TIỀN XỬ LÝ (Tạo Feature Vector)
# ==========================================
print("\n--- 2. Tạo CountVectorizer (Ma trận tần suất từ) ---")

# (Tùy chọn) Loại bỏ Stopwords tiếng Việt nếu bạn chưa làm ở bước trước
# remover = StopWordsRemover(inputCol="tokens_array", outputCol="filtered_tokens", stopWords=["của", "là", "và", "có", "được"])
# df = remover.transform(df)

# Chuyển mảng từ thành Vector tần suất (TF - Term Frequency)
# vocabSize: Giới hạn 10,000 từ phổ biến nhất để tối ưu RAM
cv = CountVectorizer(inputCol="tokens_array", outputCol="features", vocabSize=10000, minDF=5.0)
cv_model = cv.fit(df)
df_features = cv_model.transform(df)

# Lưu lại từ điển (Vocabulary) để lát nữa dịch ngược index thành chữ
vocab = cv_model.vocabulary

# ==========================================
# 3. HUẤN LUYỆN MÔ HÌNH LDA
# ==========================================
print(f"\n--- 3. Bắt đầu huấn luyện LDA với {NUM_TOPICS} Chủ đề ---")
lda = LDA(k=NUM_TOPICS, maxIter=MAX_ITER, featuresCol="features")
lda_model = lda.fit(df_features)

print("✅ Huấn luyện LDA hoàn tất.")




--- 2. Tạo CountVectorizer (Ma trận tần suất từ) ---

--- 3. Bắt đầu huấn luyện LDA với 10 Chủ đề ---
✅ Huấn luyện LDA hoàn tất.


In [5]:
# ==========================================
# 3. HUẤN LUYỆN MÔ HÌNH LDA
# ==========================================
print(f"\n--- 3. Bắt đầu huấn luyện LDA với {NUM_TOPICS} Chủ đề ---")
lda = LDA(k=NUM_TOPICS, maxIter=MAX_ITER, featuresCol="features")
lda_model = lda.fit(df_features)

print("✅ Huấn luyện LDA hoàn tất.")


--- 3. Bắt đầu huấn luyện LDA với 10 Chủ đề ---
✅ Huấn luyện LDA hoàn tất.


In [6]:
# ==========================================
# 4. TRÍCH XUẤT TỪ KHÓA & "TÓM TẮT" VĂN BẢN
# ==========================================
print("\n--- 4. Trích xuất từ khóa cho từng Chủ đề ---")
topics = lda_model.describeTopics(maxTermsPerTopic=NUM_KEYWORDS).collect()

# Tạo một Dictionary lưu trữ: {Topic_ID: "Từ_khóa_1, Từ_khóa_2,..."}
topic_summary_dict = {}
for row in topics:
    topic_id = row['topic']
    term_indices = row['termIndices']
    words = [vocab[idx] for idx in term_indices]
    topic_summary_dict[topic_id] = ", ".join(words)
    print(f"Chủ đề {topic_id}: {topic_summary_dict[topic_id]}")



--- 4. Trích xuất từ khóa cho từng Chủ đề ---
Chủ đề 0: dividend, stocks, growth, inc, earnings
Chủ đề 1: stock, ai, billion, growth, market
Chủ đề 2: earnings, stock, inc, competitors, compared
Chủ đề 3: stock, year, price, share, recent
Chủ đề 4: stock, dow, stocks, oil, futures
Chủ đề 5: target, price, maintains, nyse, analyst
Chủ đề 6: ai, stocks, growth, energy, new
Chủ đề 7: today, announced, new, ai, market
Chủ đề 8: market, stocks, trump, earnings, trading
Chủ đề 9: earnings, quarter, stocks, company, stock


In [7]:
# ==========================================
# 5. DỰ ĐOÁN VÀ GHI KẾT QUẢ VÀO ICEBERG
# ==========================================
print("\n--- 5. Phân loại bài báo và gắn Tóm tắt ---")
# Gắn cột phân bố chủ đề (topicDistribution) vào DataFrame
df_result = lda_model.transform(df_features)

# Hàm UDF để tìm ra Chủ đề chiếm tỷ trọng cao nhất trong bài báo
@udf(returnType=IntegerType())
def get_dominant_topic(topic_distribution):
    return int(np.argmax(topic_distribution))

# Hàm UDF để lấy chuỗi từ khóa tóm tắt dựa trên ID Chủ đề
@udf(returnType=StringType())
def get_topic_summary(topic_id):
    return topic_summary_dict.get(topic_id, "Không xác định")

# Áp dụng UDF
df_result = df_result.withColumn("dominant_topic_id", get_dominant_topic(col("topicDistribution")))
df_result = df_result.withColumn("lda_summary_keywords", get_topic_summary(col("dominant_topic_id")))

# Xóa các cột trung gian không cần thiết trước khi lưu
df_final = df_result.drop("tokens_array", "features", "topicDistribution")

# Đặt tên bảng Iceberg mới
output_table = "my_catalog.processed_zone_finnhub.news_lda_summarized"

print(f"⏳ Đang ghi dữ liệu vào Apache Iceberg tại: {output_table} ...")
df_final.write.format("iceberg").mode("overwrite").saveAsTable(output_table)

print("🎉 HOÀN TẤT! Các báo cáo đã được tóm tắt bằng từ khóa chủ đề và lưu vào Iceberg.")


--- 5. Phân loại bài báo và gắn Tóm tắt ---
⏳ Đang ghi dữ liệu vào Apache Iceberg tại: my_catalog.processed_zone_finnhub.news_lda_summarized ...
🎉 HOÀN TẤT! Các báo cáo đã được tóm tắt bằng từ khóa chủ đề và lưu vào Iceberg.


In [8]:
print("\n--- In ra 30 dòng đầu tiên của kết quả ---")
# Chọn các cột quan trọng để hiển thị dễ nhìn, truncate=False để không bị cắt chữ
df_final.select("title_tokens", "dominant_topic_id", "lda_summary_keywords").show(30, truncate=False)


--- In ra 30 dòng đầu tiên của kết quả ---
+----------------------------------------------------------------------------------------------------------------------------------------+-----------------+----------------------------------------+
|title_tokens                                                                                                                            |dominant_topic_id|lda_summary_keywords                    |
+----------------------------------------------------------------------------------------------------------------------------------------+-----------------+----------------------------------------+
|[market, chatter, jpmorgan, google, firms, asked, preserve, records, epstein, ranch, investigation]                                     |8                |market, stocks, trump, earnings, trading|
|[uk, regulator, sets, new, rules, google, search, boost, competition]                                                                   |7                |today, a

In [9]:
# Tên bảng bạn đã lưu kết quả LDA trước đó
table_name = "my_catalog.processed_zone_finnhub.news_lda_summarized"

print(f"--- 2. Đọc bảng dữ liệu hiện tại từ {table_name} ---")
df = spark.table(table_name)

# Từ điển ánh xạ ID sang Tên chủ đề
topic_names_dict = {
    0: "Cổ phiếu Cổ tức & Tăng trưởng",
    1: "Xu hướng AI & Vốn hóa tỷ đô",
    2: "So sánh Năng lực Cạnh tranh",
    3: "Phân tích Biến động Giá Cổ phiếu",
    4: "Chỉ số Vĩ mô & Hàng hóa",
    5: "Khuyến nghị từ Chuyên gia",
    6: "Đầu tư Năng lượng & Công nghệ Mới",
    7: "Tin tức Sự kiện & Ra mắt trong ngày",
    8: "Tác động của Yếu tố Chính trị - Vĩ mô",
    9: "Mùa Báo cáo Tài chính Quý"
}

print("--- 3. Thêm cột 'topic_name' ---")
# Hàm UDF để map dữ liệu
@udf(returnType=StringType())
def get_topic_name(topic_id):
    return topic_names_dict.get(topic_id, "Chủ đề Khác")

# Thêm cột mới dựa trên cột dominant_topic_id đã có sẵn trong bảng
df_updated = df.withColumn("topic_name", get_topic_name(col("dominant_topic_id")))

print("--- 4. Cập nhật (Ghi đè) lại bảng Iceberg ---")
# Dùng mode("overwrite") để lưu đè lại bảng cũ với schema mới (có thêm cột)
df_updated.write.format("iceberg").mode("overwrite").saveAsTable(table_name)

print("🎉 HOÀN TẤT! Đã bổ sung thành công cột Tên chủ đề vào bảng dữ liệu của bạn.")

# In ra 10 dòng xem thử thành quả
spark.table(table_name).select("title_tokens", "dominant_topic_id", "topic_name").show(10, truncate=False)


--- 2. Đọc bảng dữ liệu hiện tại từ my_catalog.processed_zone_finnhub.news_lda_summarized ---
--- 3. Thêm cột 'topic_name' ---
--- 4. Cập nhật (Ghi đè) lại bảng Iceberg ---
🎉 HOÀN TẤT! Đã bổ sung thành công cột Tên chủ đề vào bảng dữ liệu của bạn.
+----------------------------------------------------------------------------------------------------------------------------------------+-----------------+-------------------------------------+
|title_tokens                                                                                                                            |dominant_topic_id|topic_name                           |
+----------------------------------------------------------------------------------------------------------------------------------------+-----------------+-------------------------------------+
|[market, chatter, jpmorgan, google, firms, asked, preserve, records, epstein, ranch, investigation]                                     |8                |Tác động củ